# Stablecoin StressBench — Data Exploration

This notebook explores the raw Bronze and Silver data layers:
- Trade and book data quality
- Stablecoin peg deviation over time
- Cross-venue fragmentation
- On-chain settlement activity

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

DATA_DIR = Path('../data')
SILVER_DIR = DATA_DIR / 'silver'
GOLD_DIR = DATA_DIR / 'gold'

print('Data directories:')
for d in [DATA_DIR, SILVER_DIR, GOLD_DIR]:
    print(f'  {d}: exists={d.exists()}')

## 1. Trade Data Quality

In [ ]:
# Load Silver trade data
trade_files = list(SILVER_DIR.glob('trades/**/*.parquet'))
print(f'Found {len(trade_files)} trade Parquet files')

if trade_files:
    trades = pl.read_parquet(str(SILVER_DIR / 'trades' / '*.parquet'))
    print(trades.describe())
else:
    print('No trade data found. Run scripts/pull_data.py first.')

## 2. Stablecoin Peg Deviation

In [ ]:
# Load Gold stablecoin FX features
fx_files = list(GOLD_DIR.glob('feat_stablecoin_fx_1s/**/*.parquet'))
print(f'Found {len(fx_files)} FX feature files')

if fx_files:
    fx = pl.read_parquet(str(GOLD_DIR / 'feat_stablecoin_fx_1s' / '*.parquet'))
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    for ax, sc in zip(axes.flat, ['USDC', 'USDT', 'DAI', 'PYUSD']):
        sub = fx.filter(pl.col('stablecoin') == sc)
        if not sub.is_empty():
            ax.plot(sub['ts'].to_list(), sub['deviation_from_1_usd_bps'].to_list(), lw=0.5)
            ax.axhline(0, color='black', lw=0.5)
            ax.axhline(25, color='red', lw=0.5, ls='--', label='±25bps threshold')
            ax.axhline(-25, color='red', lw=0.5, ls='--')
            ax.set_title(sc)
            ax.set_ylabel('Deviation (bps)')
    plt.tight_layout()
    plt.show()
else:
    print('No FX feature data found. Run scripts/build_features.py first.')

## 3. Cross-Venue Fragmentation

In [ ]:
frag_files = list(GOLD_DIR.glob('feat_fragmentation_1m/**/*.parquet'))
if frag_files:
    frag = pl.read_parquet(str(GOLD_DIR / 'feat_fragmentation_1m' / '*.parquet'))
    print(frag.describe())
else:
    print('No fragmentation data found.')